# Predykcja severity wypadków drogowych - Baseline Model

**Dataset**: Montgomery County, Maryland - Crash Reporting (2015-2024)  
**Problem**: Klasyfikacja wieloklasowa - przewidywanie stopnia obrażeń w wypadkach drogowych  
**Target**: `Injury Severity` → uproszczony do 3 klas (NO_INJURY / MINOR / SERIOUS)

---

## 1. Wczytanie danych i przegląd

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

df = pd.read_csv("../data/01_raw/crash_data.csv")
print(f"Rozmiar: {df.shape[0]} wierszy x {df.shape[1]} kolumn")
df.head()

In [ ]:
print("Typy danych:")
print(df.dtypes)
print(f"\nKolumny ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col} - {df[col].dtype} - {df[col].nunique()} unikalnych - {df[col].isna().sum()} brakujących ({df[col].isna().mean()*100:.1f}%)")

In [ ]:
df.describe()

## 2. Eksploracja danych (EDA)

### 2.1 Zmienna docelowa - Injury Severity

In [ ]:
# Rozkład oryginalnego targetu (5 klas)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

severity_counts = df["Injury Severity"].value_counts()
severity_counts.plot(kind="bar", ax=axes[0], color=sns.color_palette("YlOrRd", len(severity_counts)))
axes[0].set_title("Injury Severity - oryginalne 5 klas")
axes[0].set_ylabel("Liczba przypadków")
axes[0].tick_params(axis="x", rotation=30)

for i, (idx, val) in enumerate(severity_counts.items()):
    axes[0].text(i, val + 500, f"{val}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)

# Uproszczony target (3 klasy)
severity_map = {
    "NO APPARENT INJURY": "NO_INJURY",
    "POSSIBLE INJURY": "MINOR",
    "SUSPECTED MINOR INJURY": "MINOR",
    "SUSPECTED SERIOUS INJURY": "SERIOUS",
    "FATAL INJURY": "SERIOUS",
}
df["Severity_Group"] = df["Injury Severity"].map(severity_map)

group_counts = df["Severity_Group"].value_counts()
colors = {"NO_INJURY": "#2ecc71", "MINOR": "#f39c12", "SERIOUS": "#e74c3c"}
group_counts.plot(kind="bar", ax=axes[1], color=[colors[x] for x in group_counts.index])
axes[1].set_title("Severity Group - uproszczone 3 klasy")
axes[1].set_ylabel("Liczba przypadków")

for i, (idx, val) in enumerate(group_counts.items()):
    axes[1].text(i, val + 500, f"{val}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("\nRozkład uproszczony:")
print(group_counts)

### 2.2 Brakujące dane

In [ ]:
# Heatmapa brakujących danych
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct}).query("count > 0").sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
missing_df["pct"].plot(kind="barh", ax=ax, color=sns.color_palette("Reds_r", len(missing_df)))
ax.set_xlabel("% brakujących")
ax.set_title("Brakujące wartości w zbiorze danych")
for i, (idx, row) in enumerate(missing_df.iterrows()):
    ax.text(row["pct"] + 0.3, i, f'{row["pct"]:.1f}% ({int(row["count"])})', va="center", fontsize=9)
plt.tight_layout()
plt.show()

### 2.3 Analiza kluczowych cech vs severity

In [ ]:
# Severity vs Weather
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Weather vs Severity
weather_sev = pd.crosstab(df["Weather"], df["Severity_Group"], normalize="index")
top_weather = df["Weather"].value_counts().head(8).index
weather_sev.loc[top_weather].plot(kind="bar", stacked=True, ax=axes[0, 0],
    color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0, 0].set_title("Severity vs Warunki pogodowe")
axes[0, 0].set_ylabel("Proporcja")
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].legend(title="Severity")

# 2. Light vs Severity
light_sev = pd.crosstab(df["Light"], df["Severity_Group"], normalize="index")
light_sev.plot(kind="bar", stacked=True, ax=axes[0, 1],
    color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[0, 1].set_title("Severity vs Oświetlenie")
axes[0, 1].set_ylabel("Proporcja")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].legend(title="Severity")

# 3. Speed Limit vs Severity
speed_data = df[df["Speed Limit"].notna() & (df["Speed Limit"] > 0)]
sns.boxplot(data=speed_data, x="Severity_Group", y="Speed Limit", ax=axes[1, 0],
    order=["NO_INJURY", "MINOR", "SERIOUS"],
    palette={"NO_INJURY": "#2ecc71", "MINOR": "#f39c12", "SERIOUS": "#e74c3c"})
axes[1, 0].set_title("Speed Limit vs Severity")

# 4. Collision Type vs Severity
collision_sev = pd.crosstab(df["Collision Type"], df["Severity_Group"], normalize="index")
top_collision = df["Collision Type"].value_counts().head(8).index
collision_sev.loc[top_collision].plot(kind="bar", stacked=True, ax=axes[1, 1],
    color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[1, 1].set_title("Severity vs Typ kolizji")
axes[1, 1].set_ylabel("Proporcja")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].legend(title="Severity")

plt.tight_layout()
plt.show()

In [ ]:
# Analiza czasowa wypadków
df["Crash Date/Time"] = pd.to_datetime(df["Crash Date/Time"], format="mixed")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Częstotliwość wg godziny
hour_counts = df.groupby([df["Crash Date/Time"].dt.hour, "Severity_Group"]).size().unstack(fill_value=0)
hour_counts.plot(kind="area", stacked=True, ax=axes[0],
    color=["#2ecc71", "#f39c12", "#e74c3c"], alpha=0.7)
axes[0].set_title("Wypadki wg godziny dnia")
axes[0].set_xlabel("Godzina")
axes[0].set_ylabel("Liczba wypadków")

# 2. Częstotliwość wg dnia tygodnia
dow_counts = df.groupby([df["Crash Date/Time"].dt.dayofweek, "Severity_Group"]).size().unstack(fill_value=0)
dow_counts.index = ["Pon", "Wt", "Śr", "Czw", "Pt", "Sob", "Nd"]
dow_counts.plot(kind="bar", stacked=True, ax=axes[1],
    color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[1].set_title("Wypadki wg dnia tygodnia")
axes[1].set_ylabel("Liczba wypadków")
axes[1].tick_params(axis="x", rotation=0)

# 3. Częstotliwość wg miesiąca
month_counts = df.groupby([df["Crash Date/Time"].dt.month, "Severity_Group"]).size().unstack(fill_value=0)
month_counts.index = ["Sty", "Lut", "Mar", "Kwi", "Maj", "Cze", "Lip", "Sie", "Wrz", "Paź", "Lis", "Gru"]
month_counts.plot(kind="bar", stacked=True, ax=axes[2],
    color=["#2ecc71", "#f39c12", "#e74c3c"])
axes[2].set_title("Wypadki wg miesiąca")
axes[2].set_ylabel("Liczba wypadków")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Mapa geograficzna wypadków wg severity
fig, ax = plt.subplots(figsize=(12, 10))
sample = df.dropna(subset=["Latitude", "Longitude"]).sample(n=min(20000, len(df)), random_state=42)

for severity, color, alpha, size in [
    ("NO_INJURY", "#2ecc71", 0.1, 1),
    ("MINOR", "#f39c12", 0.3, 3),
    ("SERIOUS", "#e74c3c", 0.8, 15),
]:
    mask = sample["Severity_Group"] == severity
    ax.scatter(sample.loc[mask, "Longitude"], sample.loc[mask, "Latitude"],
        c=color, s=size, alpha=alpha, label=severity)

ax.set_title("Mapa wypadków - Montgomery County, MD")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
# Usunięcie kolumn nieprzydatnych / o zbyt wielu brakujących wartościach
columns_to_drop = [
    "Report Number", "Local Case Number", "Agency Name",
    "Person ID", "Vehicle ID", "Location",
    "Off-Road Description",  # 91% brakujących
    "Related Non-Motorist",  # 97% brakujących
    "Non-Motorist Substance Abuse",  # 97% brakujących
    "ACRS Report Type",
    "Injury Severity",  # zastąpione przez Severity_Group
]
df_clean = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
print(f"Po usunięciu kolumn: {df_clean.shape}")

# Feature engineering z daty
df_clean["crash_hour"] = df_clean["Crash Date/Time"].dt.hour
df_clean["crash_dayofweek"] = df_clean["Crash Date/Time"].dt.dayofweek
df_clean["crash_month"] = df_clean["Crash Date/Time"].dt.month
df_clean["crash_year"] = df_clean["Crash Date/Time"].dt.year
df_clean = df_clean.drop(columns=["Crash Date/Time"])

# Cechy pochodne
df_clean["is_night"] = df_clean["Light"].str.contains("DARK", case=False, na=False).astype(int)
df_clean["is_bad_weather"] = (~df_clean["Weather"].isin(["CLEAR", "CLOUDY"])).astype(int)
df_clean["is_wet_surface"] = (~df_clean["Surface Condition"].isin(["DRY"])).astype(int)

if "Vehicle Year" in df_clean.columns:
    df_clean["vehicle_age"] = 2026 - df_clean["Vehicle Year"]
    df_clean["vehicle_age"] = df_clean["vehicle_age"].clip(lower=0, upper=50)

print(f"Po feature engineering: {df_clean.shape}")
df_clean.head()

In [ ]:
# Imputacja brakujących wartości
for col in df_clean.select_dtypes(include=["object"]).columns:
    if col != "Severity_Group":
        mode_val = df_clean[col].mode()
        df_clean[col] = df_clean[col].fillna(mode_val[0] if len(mode_val) > 0 else "UNKNOWN")

for col in df_clean.select_dtypes(include=["number"]).columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f"Brakujące wartości po imputacji: {df_clean.isnull().sum().sum()}")

# Label encoding cech kategorycznych (nie one-hot - wysoka kardynalność)
label_encoders = {}
for col in df_clean.select_dtypes(include=["object"]).columns:
    if col == "Severity_Group":
        continue
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    
print(f"\nZakodowane kolumny: {list(label_encoders.keys())}")
print(f"Końcowy kształt danych: {df_clean.shape}")
df_clean.dtypes

## 4. Trening modelu bazowego

In [ ]:
# Podział na zbiór treningowy i testowy
X = df_clean.drop("Severity_Group", axis=1)
y = df_clean["Severity_Group"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Zbiór treningowy: {X_train.shape}")
print(f"Zbiór testowy:    {X_test.shape}")
print(f"\nRozkład klas w zbiorze treningowym:")
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# Model 1: Random Forest (z class_weight='balanced')
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("=" * 60)
print("RANDOM FOREST (class_weight='balanced')")
print("=" * 60)
print(classification_report(y_test, rf_preds))
print(f"Accuracy:    {accuracy_score(y_test, rf_preds):.4f}")
print(f"F1 weighted: {f1_score(y_test, rf_preds, average='weighted'):.4f}")
print(f"F1 macro:    {f1_score(y_test, rf_preds, average='macro'):.4f}")

In [ ]:
# Model 2: Logistic Regression (sanity check baseline)
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    multi_class="multinomial",
)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print("=" * 60)
print("LOGISTIC REGRESSION (class_weight='balanced')")
print("=" * 60)
print(classification_report(y_test, lr_preds))
print(f"Accuracy:    {accuracy_score(y_test, lr_preds):.4f}")
print(f"F1 weighted: {f1_score(y_test, lr_preds, average='weighted'):.4f}")
print(f"F1 macro:    {f1_score(y_test, lr_preds, average='macro'):.4f}")

## 5. Ewaluacja

In [ ]:
# Confusion matrix dla obu modeli
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
labels = ["MINOR", "NO_INJURY", "SERIOUS"]

for ax, preds, title in [
    (axes[0], rf_preds, "Random Forest"),
    (axes[1], lr_preds, "Logistic Regression"),
]:
    cm = confusion_matrix(y_test, preds, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
        xticklabels=labels, yticklabels=labels)
    ax.set_title(f"Confusion Matrix - {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (Random Forest)
feature_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
feature_imp.head(20).plot(kind="barh", ax=ax, color=sns.color_palette("viridis", 20))
ax.set_title("Top 20 najważniejszych cech (Random Forest)")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 cech:")
for i, (feat, imp) in enumerate(feature_imp.head(10).items()):
    print(f"  {i+1}. {feat}: {imp:.4f}")

In [ ]:
# Podsumowanie porównania modeli
results = pd.DataFrame({
    "Model": ["Random Forest", "Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test, rf_preds),
        accuracy_score(y_test, lr_preds),
    ],
    "F1 Weighted": [
        f1_score(y_test, rf_preds, average="weighted"),
        f1_score(y_test, lr_preds, average="weighted"),
    ],
    "F1 Macro": [
        f1_score(y_test, rf_preds, average="macro"),
        f1_score(y_test, lr_preds, average="macro"),
    ],
})
results = results.round(4)
print("Porównanie modeli baseline:")
results

## Wnioski

1. **Silny class imbalance** - klasa SERIOUS stanowi ~1% danych, co utrudnia jej detekcję
2. **Random Forest** z `class_weight='balanced'` daje lepsze wyniki niż Logistic Regression
3. **Najważniejsze cechy**: Vehicle Damage Extent, Collision Type, Speed Limit, Vehicle Movement
4. **Kierunki ulepszania**:
   - SMOTE / oversampling dla klasy SERIOUS
   - Gradient Boosting (XGBoost, LightGBM)
   - Optymalizacja hiperparametrów (Optuna)
   - Lepszy feature engineering (interakcje cech, target encoding)